In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.4 MB/s eta 0:00:00


In [3]:
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/final-year-research"
WORK_DIR = "/content/final-year-research"
ZIP_PATH = f"{PROJECT_DIR}/dataset/WilderPerson.zip"

os.makedirs(WORK_DIR, exist_ok=True)
shutil.copy(ZIP_PATH, f"{WORK_DIR}/WilderPerson.zip")
print("Copied zip to Colab VM")

Copied zip to Colab VM


In [4]:
import zipfile

with zipfile.ZipFile(f"{WORK_DIR}/WilderPerson.zip", 'r') as zip_ref:
    zip_ref.extractall(f"{WORK_DIR}/dataset")

print("Unzipped dataset")

Unzipped dataset


In [5]:
!pip install scikit-image

In [ ]:
import cv2
from skimage.metrics import structural_similarity as ssim

original_path = "/content/drive/MyDrive/final-year-research/datasets/gaussian_faceblur_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

# Check if images loaded
if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    # make sure shapes match
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    score = ssim(gray1, gray2)
    privacy_score = 1 - score

    print("SSIM:", score)
    print("Privacy Score:", privacy_score)

SSIM: 0.9948279889808536
Privacy Score: 0.005172011019146439


In [ ]:
import cv2
from skimage.metrics import structural_similarity as ssim

original_path = "/content/drive/MyDrive/final-year-research/transformed_datasets/faceblur_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

# Check if images loaded
if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    # make sure shapes match
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    score = ssim(gray1, gray2)
    privacy_score = 1 - score

    print("SSIM:", score)
    print("Privacy Score:", privacy_score)

SSIM: 0.9912067591350892
Privacy Score: 0.008793240864910845


In [ ]:
import cv2
from skimage.metrics import structural_similarity as ssim

# Paths
original_path = "/content/drive/MyDrive/final-year-research/transformed_datasets/faceblur_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

label_path = "/content/final-year-research/dataset/train/labels/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.txt"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    scores = []

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width / 2) * w_img)
        y = int((y_center - height / 2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        # Skip very small regions
        min_dim = min(gray1.shape[0], gray1.shape[1])
        if min_dim < 7:
            continue

        # Adjust win_size if necessary
        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)
        scores.append(score)

    if scores:
        avg_ssim = sum(scores) / len(scores)
        privacy_score = 1 - avg_ssim

        print("Head Region SSIM:", avg_ssim)
        print("Head Region Privacy Score:", privacy_score)
    else:
        print("No valid head regions found")

Head Region SSIM: 0.7841729792002089
Head Region Privacy Score: 0.2158270207997911


In [ ]:
import cv2
from skimage.metrics import structural_similarity as ssim

# Paths
original_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

label_path = "/content/final-year-research/dataset/train/labels/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.txt"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    scores = []

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width / 2) * w_img)
        y = int((y_center - height / 2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        # Skip very small regions
        min_dim = min(gray1.shape[0], gray1.shape[1])
        if min_dim < 7:
            continue

        # Adjust win_size if necessary
        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)
        scores.append(score)

    if scores:
        avg_ssim = sum(scores) / len(scores)
        privacy_score = 1 - avg_ssim

        print("Head Region SSIM:", avg_ssim)
        print("Head Region Privacy Score:", privacy_score)
    else:
        print("No valid head regions found")

Head Region SSIM: 1.0
Head Region Privacy Score: 0.0


In [15]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

# Dataset paths
original_dir = "/content/final-year-research/dataset/valid/images"
anonymized_dir = "/content/drive/MyDrive/final-year-research/datasets/gaussian_faceblur_dataset/valid/images"
labels_dir = "/content/final-year-research/dataset/valid/labels"

scores = []
total_heads = 0

image_files = os.listdir(original_dir)

for img_name in image_files:

    img_path1 = os.path.join(original_dir, img_name)
    img_path2 = os.path.join(anonymized_dir, img_name)
    label_path = os.path.join(labels_dir, img_name.replace(".jpg", ".txt"))

    if not os.path.exists(label_path):
        continue

    img1 = cv2.imread(img_path1)
    img2 = cv2.imread(img_path2)

    if img1 is None or img2 is None:
        continue

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width/2) * w_img)
        y = int((y_center - height/2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        min_dim = min(gray1.shape)

        if min_dim < 7:
            continue

        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)

        scores.append(score)
        total_heads += 1


if scores:

    avg_ssim = np.mean(scores)
    privacy_score = 1 - avg_ssim

    print("Total head regions evaluated:", total_heads)
    print("Average Head SSIM:", avg_ssim)
    print("Average Privacy Score:", privacy_score)

else:
    print("No valid regions found")

Total head regions evaluated: 152340
Average Head SSIM: 0.7775808948253973
Average Privacy Score: 0.22241910517460273


# **Evaluate Pixelated Privacy score**

In [6]:
import cv2
from skimage.metrics import structural_similarity as ssim

original_path = "/content/drive/MyDrive/final-year-research/datasets/pixelated_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

# Check if images loaded
if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    # make sure shapes match
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    score = ssim(gray1, gray2)
    privacy_score = 1 - score

    print("SSIM:", score)
    print("Privacy Score:", privacy_score)

SSIM: 0.9950654605594512
Privacy Score: 0.00493453944054878


In [8]:
import cv2
from skimage.metrics import structural_similarity as ssim

# Paths
original_path = "/content/drive/MyDrive/final-year-research/datasets/pixelated_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

label_path = "/content/final-year-research/dataset/train/labels/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.txt"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    scores = []

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width / 2) * w_img)
        y = int((y_center - height / 2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        # Skip very small regions
        min_dim = min(gray1.shape[0], gray1.shape[1])
        if min_dim < 7:
            continue

        # Adjust win_size if necessary
        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)
        scores.append(score)

    if scores:
        avg_ssim = sum(scores) / len(scores)
        privacy_score = 1 - avg_ssim

        print("Head Region SSIM:", avg_ssim)
        print("Head Region Privacy Score:", privacy_score)
    else:
        print("No valid head regions found")

Head Region SSIM: 0.8770664132727564
Head Region Privacy Score: 0.12293358672724364


In [9]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

# Dataset paths
original_dir = "/content/final-year-research/dataset/valid/images"
anonymized_dir = "/content/drive/MyDrive/final-year-research/datasets/pixelated_dataset/valid/images"
labels_dir = "/content/final-year-research/dataset/valid/labels"

scores = []
total_heads = 0

image_files = os.listdir(original_dir)

for img_name in image_files:

    img_path1 = os.path.join(original_dir, img_name)
    img_path2 = os.path.join(anonymized_dir, img_name)
    label_path = os.path.join(labels_dir, img_name.replace(".jpg", ".txt"))

    if not os.path.exists(label_path):
        continue

    img1 = cv2.imread(img_path1)
    img2 = cv2.imread(img_path2)

    if img1 is None or img2 is None:
        continue

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width/2) * w_img)
        y = int((y_center - height/2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        min_dim = min(gray1.shape)

        if min_dim < 7:
            continue

        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)

        scores.append(score)
        total_heads += 1


if scores:

    avg_ssim = np.mean(scores)
    privacy_score = 1 - avg_ssim

    print("Total head regions evaluated:", total_heads)
    print("Average Head SSIM:", avg_ssim)
    print("Average Privacy Score:", privacy_score)

else:
    print("No valid regions found")

Total head regions evaluated: 152340
Average Head SSIM: 0.792753322985561
Average Privacy Score: 0.207246677014439


# **Face Mask Privacy Evaluation**

In [11]:
import cv2
from skimage.metrics import structural_similarity as ssim

original_path = "/content/drive/MyDrive/final-year-research/datasets/facemask_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

# Check if images loaded
if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    # make sure shapes match
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    score = ssim(gray1, gray2)
    privacy_score = 1 - score

    print("SSIM:", score)
    print("Privacy Score:", privacy_score)

SSIM: 0.9912067591350892
Privacy Score: 0.008793240864910845


In [13]:
import cv2
from skimage.metrics import structural_similarity as ssim

# Paths
original_path = "/content/drive/MyDrive/final-year-research/datasets/facemask_dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"
anonymized_path = "/content/final-year-research/dataset/train/images/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.jpg"

label_path = "/content/final-year-research/dataset/train/labels/000040_jpg.rf.64f30d73c5a26032ce43773ddd891638.txt"

img1 = cv2.imread(original_path)
img2 = cv2.imread(anonymized_path)

if img1 is None:
    print("Error: Original image not found")
if img2 is None:
    print("Error: Anonymized image not found")

if img1 is not None and img2 is not None:

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    scores = []

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width / 2) * w_img)
        y = int((y_center - height / 2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        # Skip very small regions
        min_dim = min(gray1.shape[0], gray1.shape[1])
        if min_dim < 7:
            continue

        # Adjust win_size if necessary
        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)
        scores.append(score)

    if scores:
        avg_ssim = sum(scores) / len(scores)
        privacy_score = 1 - avg_ssim

        print("Head Region SSIM:", avg_ssim)
        print("Head Region Privacy Score:", privacy_score)
    else:
        print("No valid head regions found")

Head Region SSIM: 0.7841729792002089
Head Region Privacy Score: 0.2158270207997911


In [14]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

# Dataset paths
original_dir = "/content/final-year-research/dataset/valid/images"
anonymized_dir = "/content/drive/MyDrive/final-year-research/datasets/facemask_dataset/valid/images"
labels_dir = "/content/final-year-research/dataset/valid/labels"

scores = []
total_heads = 0

image_files = os.listdir(original_dir)

for img_name in image_files:

    img_path1 = os.path.join(original_dir, img_name)
    img_path2 = os.path.join(anonymized_dir, img_name)
    label_path = os.path.join(labels_dir, img_name.replace(".jpg", ".txt"))

    if not os.path.exists(label_path):
        continue

    img1 = cv2.imread(img_path1)
    img2 = cv2.imread(img_path2)

    if img1 is None or img2 is None:
        continue

    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    h_img, w_img = img1.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        class_id, x_center, y_center, width, height = map(float, line.split())

        # YOLO → pixel coordinates
        x = int((x_center - width/2) * w_img)
        y = int((y_center - height/2) * h_img)
        w = int(width * w_img)
        h = int(height * h_img)

        crop1 = img1[y:y+h, x:x+w]
        crop2 = img2[y:y+h, x:x+w]

        if crop1.size == 0 or crop2.size == 0:
            continue

        gray1 = cv2.cvtColor(crop1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(crop2, cv2.COLOR_BGR2GRAY)

        min_dim = min(gray1.shape)

        if min_dim < 7:
            continue

        win_size = min(7, min_dim)
        if win_size % 2 == 0:
            win_size -= 1

        score = ssim(gray1, gray2, win_size=win_size)

        scores.append(score)
        total_heads += 1


if scores:

    avg_ssim = np.mean(scores)
    privacy_score = 1 - avg_ssim

    print("Total head regions evaluated:", total_heads)
    print("Average Head SSIM:", avg_ssim)
    print("Average Privacy Score:", privacy_score)

else:
    print("No valid regions found")

Total head regions evaluated: 152340
Average Head SSIM: 0.6648999184560112
Average Privacy Score: 0.33510008154398885
